# Week 15, LangGraph Support Agent (State-Graph Model)

```text
# Requirements: pip install langgraph openai
```

> ⚠️ REQUIRES: `OPENAI_API_KEY` or `OPENROUTER_API_KEY` (for the real-model answer hook). With no key, a deterministic classifier drives the same graph, so every cell still runs.
> ⚠️ REQUIRES: `langgraph` installed. If it is missing, a manual runner executes the *same node functions* in the *same graph order*, so the per-node table and score still print.

Re-implement the Week 14 agent as an explicit **state graph**: a `TypedDict` state, nodes for `intent → tool → generate → escalate` (plus a human `refund_approval` gate), conditional edges, `MemorySaver` checkpoints, and `interrupt_before` for refunds over $500.

## The state-graph model

Where Week 14 carried state implicitly in a message list, LangGraph makes it a **named, typed contract** that flows through **nodes** (functions) along **edges** (transitions). That naming is what unlocks the framework's three superpowers: **conditional routing** (the router returns a key and the graph decides), **checkpointing** (`MemorySaver` persists state after every node so a run can be *resumed*), and **interrupts** (`interrupt_before` pauses before a named node for human approval).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
_root = pathlib.Path.cwd()
while not (_root / "zoro").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import json, os, re, time
from typing import TypedDict
import numpy as np
from zoro import data

SEED = 42
rng = np.random.default_rng(SEED)

HAS_LG = False
try:
    from langgraph.graph import StateGraph, START, END
    from langgraph.checkpoint.memory import MemorySaver
    try:
        from langgraph.types import Command
    except Exception:
        Command = None
    HAS_LG = True
    print("langgraph imported OK")
except Exception as e:
    print("langgraph not available (", type(e).__name__, "); a manual runner will execute the same nodes.")

_ship = data.shipments(n=50_000, seed=SEED)
_lanes = data.lanes(seed=11)
_car = data.carriers(seed=7)
_ship_by_id = {row.shipment_id: row for row in _ship.itertuples()}
_lane_by_id = {row.lane_id: row for row in _lanes.itertuples()}
_car_by_id = {row.carrier_id: row for row in _car.itertuples()}
_policies = {d["doc_id"]: d for d in data.policy_docs()}

class AgentState(TypedDict):
    ticket_text: str
    shipment_id: str
    intent: str
    refund_amount: float
    needs_approval: bool
    tool_result: str
    final_answer: str
    escalated: bool
    history: list

print("state + tools ready")

## Tools & the intent classifier

Two tools (tracking + policy) and a classifier that reads a ticket and returns the intent, the shipment id, and, for refunds, the **computed refund amount** (10% if >48h late, 50% if >7 days, per `POL-002`). A refund over **$500** flips `needs_approval`, which is the signal the human gate waits on.

In [ ]:
def track_shipment(shipment_id):
    sid = str(shipment_id).strip().upper()
    row = _ship_by_id.get(sid)
    if row is None:
        return {"error": "shipment " + sid + " not found"}
    lane = _lane_by_id.get(row.lane_id)
    carrier = _car_by_id.get(row.carrier_id)
    return {
        "shipment_id": sid,
        "status": row.status,
        "carrier": carrier.carrier_name if carrier else row.carrier_id,
        "origin": lane.origin if lane else "?",
        "destination": lane.destination if lane else "?",
        "delay_hours": round(float(row.delay_hours), 1),
        "on_time": bool(row.is_on_time),
    }

def get_policy(doc_id):
    doc = _policies.get(str(doc_id).strip().upper())
    return {"doc_id": doc["doc_id"], "title": doc["title"], "text": doc["text"]} if doc else {"error": "no policy"}

def compute_refund(shipment_id):
    row = _ship_by_id.get(str(shipment_id).strip().upper())
    if row is None:
        return 0.0
    delay = float(row.delay_hours)
    value = float(row.value_usd)
    if delay > 7 * 24:
        return round(value * 0.50, 2)
    if delay > 48:
        return round(value * 0.10, 2)
    return 0.0

def classify_intent(text):
    low = str(text).lower()
    sid_m = re.search(r"S\d{7}", str(text), re.I)
    sid = sid_m.group(0).upper() if sid_m else ""
    if "refund" in low or "damaged" in low or "damage" in low or "invoice" in low or "wrong weight" in low or "billing" in low:
        amount = compute_refund(sid)
        return {"intent": "refund", "shipment_id": sid, "refund_amount": amount, "needs_approval": amount > 500.0}
    if "bill of lading" in low or "document" in low or "customs" in low:
        return {"intent": "docs", "shipment_id": sid, "refund_amount": 0.0, "needs_approval": False}
    if "where is" in low or "track" in low or "arrive" in low or "status" in low or "late" in low:
        return {"intent": "tracking", "shipment_id": sid, "refund_amount": 0.0, "needs_approval": False}
    return {"intent": "escalate", "shipment_id": sid, "refund_amount": 0.0, "needs_approval": False}

def _llm(prompt):
    try:
        import openai
    except Exception:
        return None
    key = os.environ.get("OPENAI_API_KEY")
    base, model = None, "gpt-4o-mini"
    if not key:
        key = os.environ.get("OPENROUTER_API_KEY")
        if key:
            base, model = "https://openrouter.ai/api/v1", "openai/gpt-4o-mini"
    if not key:
        return None
    client = openai.OpenAI(api_key=key, base_url=base) if base else openai.OpenAI(api_key=key)
    r = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], temperature=0.0)
    return r.choices[0].message.content

print("classify_intent example:", classify_intent("Where is my shipment S0000123? It was supposed to arrive 2026-01-12."))

## Nodes

Each node is a plain function `(state) -> partial state`. Timing and token estimates are recorded into `NODE_STATS` so the framework's overhead and each node's cost stay visible.

In [ ]:
NODE_STATS = {k: {"latency": 0.0, "tokens": 0, "calls": 0} for k in ["intent", "tool", "generate", "escalate", "refund_approval"]}

def _est(text):
    return max(1, len(str(text)) // 4)

def _tick(name, t0):
    NODE_STATS[name]["latency"] += time.perf_counter() - t0
    NODE_STATS[name]["calls"] += 1

def _log(state, entry):
    return list(state.get("history", [])) + [entry]

def intent_node(state):
    t0 = time.perf_counter()
    info = classify_intent(state.get("ticket_text", ""))
    NODE_STATS["intent"]["tokens"] += _est(state.get("ticket_text", ""))
    _tick("intent", t0)
    return {
        "intent": info["intent"],
        "shipment_id": info.get("shipment_id", ""),
        "refund_amount": info.get("refund_amount", 0.0),
        "needs_approval": info.get("needs_approval", False),
        "history": _log(state, "intent=" + info["intent"]),
    }

def tool_node(state):
    t0 = time.perf_counter()
    intent = state.get("intent", "")
    if intent == "tracking":
        res = track_shipment(state.get("shipment_id"))
    elif intent == "docs":
        res = get_policy("POL-002")
    elif intent == "refund":
        res = {"refund": "eligible", "amount_usd": state.get("refund_amount", 0.0)}
    else:
        res = {"error": "unknown intent " + intent}
    text = json.dumps(res)
    NODE_STATS["tool"]["tokens"] += _est(text)
    _tick("tool", t0)
    return {"tool_result": text, "history": _log(state, "tool=" + text[:60])}

def generate_node(state):
    t0 = time.perf_counter()
    intent = state.get("intent", "")
    tr = state.get("tool_result", "")
    prompt = "Answer this support ticket. intent=" + intent + " tool_result=" + tr[:500]
    llm_answer = _llm(prompt)
    if llm_answer:
        answer = llm_answer
    elif intent == "tracking":
        answer = "Tracking result: " + tr
    elif intent == "docs":
        answer = "Policy lookup: " + tr
    elif intent == "refund":
        answer = "Refund processed for " + str(state.get("refund_amount", 0.0)) + " USD."
    else:
        answer = "Escalated."
    NODE_STATS["generate"]["tokens"] += _est(answer)
    _tick("generate", t0)
    return {"final_answer": answer, "history": _log(state, "generate=" + answer[:60])}

def escalate_node(state):
    t0 = time.perf_counter()
    NODE_STATS["escalate"]["tokens"] += _est("escalated")
    _tick("escalate", t0)
    return {"final_answer": "Escalated to a human agent for review.", "escalated": True, "history": _log(state, "escalate")}

def refund_approval_node(state):
    t0 = time.perf_counter()
    amount = state.get("refund_amount", 0.0)
    NODE_STATS["refund_approval"]["tokens"] += _est("approved")
    _tick("refund_approval", t0)
    return {"final_answer": "Refund of $" + str(amount) + " approved by a human and issued.", "history": _log(state, "human_approved_refund=" + str(amount))}

def route_after_intent(state):
    if state.get("intent") == "escalate":
        return "escalate"
    if state.get("needs_approval"):
        return "refund_approval"
    return "tool"

def initial_state(ticket_text, shipment_id=""):
    return {
        "ticket_text": ticket_text,
        "shipment_id": shipment_id,
        "intent": "",
        "refund_amount": 0.0,
        "needs_approval": False,
        "tool_result": "",
        "final_answer": "",
        "escalated": False,
        "history": [],
    }

print("nodes defined:", list(NODE_STATS))

## Graph assembly + visualization

Nodes become a `StateGraph`; the conditional edge maps the router's return key to the next node. The graph is printed as Mermaid (and ASCII) so the shape is inspectable.

In [ ]:
MERMAID = """graph TD
  START --> intent
  intent -->|tracking/docs| tool
  intent -->|refund > $500| refund_approval
  intent -->|escalate| escalate
  tool --> generate
  refund_approval --> generate
  generate --> END
  escalate --> END"""

print("--- Mermaid ---")
print(MERMAID)

graph = None
checkpointer = None
if HAS_LG:
    g = StateGraph(AgentState)
    g.add_node("intent", intent_node)
    g.add_node("tool", tool_node)
    g.add_node("generate", generate_node)
    g.add_node("escalate", escalate_node)
    g.add_node("refund_approval", refund_approval_node)
    g.add_edge(START, "intent")
    g.add_conditional_edges(
        "intent",
        route_after_intent,
        {"tool": "tool", "refund_approval": "refund_approval", "escalate": "escalate"},
    )
    g.add_edge("tool", "generate")
    g.add_edge("refund_approval", "generate")
    g.add_edge("generate", END)
    g.add_edge("escalate", END)
    checkpointer = MemorySaver()
    graph = g.compile(checkpointer=checkpointer, interrupt_before=["refund_approval"])
    try:
        print("--- graph.draw_mermaid() ---")
        print(graph.get_graph().draw_mermaid())
    except Exception as e:
        print("(draw_mermaid unavailable:", type(e).__name__, ")")
    print("graph compiled with MemorySaver + interrupt_before=['refund_approval']")
else:
    print("(langgraph absent; manual runner will walk the same nodes/edges)")

## Checkpoints + resume

`MemorySaver` persists state after every node. Run a tracking ticket, inspect the saved state, then run a *second* ticket on the **same thread** and confirm the `history` accumulates, that is conversation memory as durable state, not a hidden variable.

In [ ]:
def manual_run(state):
    out = intent_node(state)
    state = {**state, **out}
    r = route_after_intent(state)
    if r == "tool":
        state = {**state, **tool_node(state)}
        state = {**state, **generate_node(state)}
    elif r == "refund_approval":
        state = {**state, **refund_approval_node(state)}
        state = {**state, **generate_node(state)}
    else:
        state = {**state, **escalate_node(state)}
    return state

ticket_track = "Where is my shipment S0000123? It was supposed to arrive 2026-01-12."
if HAS_LG:
    cfg = {"configurable": {"thread_id": "t-1"}}
    out = graph.invoke(initial_state(ticket_track), cfg)
    print("run 1 final:", out["final_answer"])
    snap = graph.get_state(cfg)
    print("checkpointed history:", snap.values.get("history"))
    out2 = graph.invoke(initial_state("Is shipment S0000456 going to arrive on time?"), cfg)
    print("run 2 (same thread) history length:", len(out2.get("history", [])), "(persisted across runs)")
else:
    s = manual_run(initial_state(ticket_track))
    print("manual run final:", s["final_answer"])
    print("history:", s["history"])

## Human-in-the-loop: refund approval

Find a refund ticket whose computed refund exceeds **$500**. The graph pauses *before* `refund_approval` (that is what `interrupt_before` guards). A human approves by resuming; without that resume, no refund is issued.

In [ ]:
# Deterministic refund scenario: a real shipment with a computed refund over $500.
# S0000036 is >48h late with a high declared value, so the 10% refund exceeds $500.
refund_sid = "S0000036"
refund_amount = compute_refund(refund_sid)
refund_text = ("I want a refund for shipment " + refund_sid + ". It arrived "
               + str(int(round(float(_ship_by_id[refund_sid].delay_hours)))) + " hours late.")
print("chosen refund shipment:", refund_sid, "| refund amount:", refund_amount, "| needs approval:", refund_amount > 500.0)
print("text:", refund_text)

if HAS_LG:
    cfg2 = {"configurable": {"thread_id": "t-2"}}
    paused = False
    try:
        res = graph.invoke(initial_state(refund_text, refund_sid), cfg2)
        if res.get("__interrupt__"):
            paused = True
    except Exception as e:
        if "interrupt" in str(e).lower():
            paused = True
        else:
            raise
    print("PAUSED before refund_approval (needs human approval):", paused)
    snap2 = graph.get_state(cfg2)
    print("pending state intent:", snap2.values.get("intent"), "| needs_approval:", snap2.values.get("needs_approval"))
    if Command is not None:
        res2 = graph.invoke(Command(resume={"approved": True}), cfg2)
    else:
        res2 = graph.invoke(None, cfg2)
    print("resumed final:", res2["final_answer"])
else:
    s2 = manual_run(initial_state(refund_text, refund_sid))
    print("manual final:", s2["final_answer"])


## Per-node latency & tokens

Run 20 real tickets through the node functions and aggregate latency and tokens **per node**, the table that tells you where your agent's time and money actually go.

In [ ]:
for k in NODE_STATS:
    NODE_STATS[k] = {"latency": 0.0, "tokens": 0, "calls": 0}

sample = data.support_tickets(n=20, seed=99)
for t in sample.itertuples():
    manual_run(initial_state(t.text, t.shipment_id))

print("node          calls   total_latency(ms)   total_tokens   avg_latency(ms)")
for name, s in NODE_STATS.items():
    avg = (s["latency"] * 1000 / s["calls"]) if s["calls"] else 0.0
    print("%-13s %4d   %15.2f   %12d   %14.2f" % (name, s["calls"], s["latency"] * 1000, s["tokens"], avg))

total_latency_ms = sum(s["latency"] for s in NODE_STATS.values()) * 1000
total_tokens = sum(s["tokens"] for s in NODE_STATS.values())
print("TOTAL latency %.2f ms · tokens %d" % (total_latency_ms, total_tokens))

In [ ]:
CAT_TO_INTENT = {"tracking": "tracking", "refund": "refund", "damage": "refund", "billing": "refund", "documents": "docs", "customs": "docs"}
correct = 0
for t in sample.itertuples():
    pred = classify_intent(t.text)["intent"]
    gt = CAT_TO_INTENT.get(t.category, "escalate")
    correct += 1 if pred == gt else 0
routing_accuracy = round(correct / len(sample), 4)
print("Correctly routed:", correct, "/", len(sample))
print("ROUTING_ACCURACY", routing_accuracy)